# Perceptual MM-Cache Benchmark — Colab Pro

GPU-heavy execution layer for the `pmcache` project. **Upload this notebook and run cells in order, top to bottom.** There is ONE kernel restart in the middle (cell 8) — after it fires, just continue from the cell directly below it.

Pipeline:
1. Install vLLM + LMCache + project deps
2. **(kernel restart)**
3. Clone the repo, prepare a Video-MME subset
4. Smoke-test the VLM
5. Run baseline (vLLM only, no perceptual aliasing)
6. Run perceptual (pmcache shim aliases similar frames)
7. Compare side-by-side (accuracy + KV bytes)
8. Optional: threshold sweep, final report, Gradio demo

**Before running:** `Runtime → Change runtime type → A100` (preferred) or `L4` (24 GB). T4 (16 GB) is too small.

## Configuration
Edit these once. They get re-set after the kernel restart in a later cell — keep them consistent.

In [ ]:
# ====== EDIT THIS ======
REPO_URL  = "https://github.com/dheerajmr01/perceptual-mmcache.git"
BRANCH    = "main"
MODEL     = "Qwen/Qwen3-VL-2B-Instruct"   # 256K context, verified A100-40GB
WORKSPACE = "/content/drive/MyDrive/pmcache"
# =======================

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PMCACHE_WORKSPACE"] = WORKSPACE
os.environ["PMCACHE_DEVICE"]    = "cuda"  # DinoV2Verifier auto-picks cuda; explicit for clarity

## 0. GPU sanity check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU detected — change runtime type to A100/L4"
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
print(f"\nDevice: {props.name}")
print(f"VRAM:   {vram_gb:.1f} GB")
assert vram_gb >= 20, f"Need >=20 GB VRAM for Qwen3-VL-2B. Got {vram_gb:.1f} GB."

## 1. Mount Drive
All results and downloaded videos persist to Drive so a disconnected session doesn't lose progress.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Plain shortcuts for the pre-clone cells that can't yet import eval.paths.
BASELINE_PATH   = f'{WORKSPACE}/results/baseline.jsonl'
PERCEPTUAL_PATH = f'{WORKSPACE}/results/perceptual.jsonl'
SWEEP_DIR       = f'{WORKSPACE}/results/sweep'
REPORT_DIR      = f'{WORKSPACE}/results/report'
VIDEOS_DIR      = f'{WORKSPACE}/videos'

for d in (VIDEOS_DIR, SWEEP_DIR, REPORT_DIR):
    os.makedirs(d, exist_ok=True)

print(f"Workspace: {WORKSPACE}")

## 2. Install dependencies

vLLM pins a specific torch ABI version. Colab ships an older torch preinstalled, so the install + restart pattern below is mandatory:

1. Install `vllm + lmcache + transformers + accelerate` together (single command — splitting them can leave torch at a mismatched version, triggering `ImportError: cannot import name 'is_opaque_value'`).
2. **Restart the kernel** so the freshly-installed torch is loaded.
3. Resume from the cell below the restart marker.

In [ ]:
%%capture
!pip install -q --upgrade pip

# Install vllm + lmcache + the HF stack together so pip resolves a
# torch version compatible with all of them.
!pip install -q -U \
    vllm \
    lmcache \
    transformers \
    accelerate \
    hf_transfer

# Pure-Python / non-torch deps in their own batch — order-insensitive.
!pip install -q \
    imagehash \
    pybktree \
    opencv-python-headless \
    pillow \
    matplotlib \
    pandas \
    tqdm \
    seaborn \
    gradio \
    pytest \
    datasets \
    yt-dlp

### ⚠️ RESTART KERNEL

The next cell kills the Python kernel. Colab will show a 'Your session crashed' notice — that's expected. Just press 'Reconnect' (or refresh) and continue with the cells **below** this restart cell.

Do NOT re-run the install cells after the restart. They persist on disk.

In [ ]:
# ===== RESTART RUNTIME =====
import os
print("Restarting kernel so the newly-installed torch is loaded...")
os.kill(os.getpid(), 9)

---
# ▶ AFTER RESTART — RESUME HERE

The kernel just restarted. All Python imports and variables are gone, but installed packages, Drive mount, and the editable repo install on disk all persisted. Run each cell from here in order.

## 3. Re-set config + re-mount Drive

Variables were wiped by the restart. Mounting Drive is idempotent (returns immediately if already mounted).

In [ ]:
# Keep these in sync with the Configuration cell at the top.
REPO_URL  = "https://github.com/dheerajmr01/perceptual-mmcache.git"
BRANCH    = "main"
MODEL     = "Qwen/Qwen3-VL-2B-Instruct"
WORKSPACE = "/content/drive/MyDrive/pmcache"

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PMCACHE_WORKSPACE"] = WORKSPACE
os.environ["PMCACHE_DEVICE"]    = "cuda"

from google.colab import drive
drive.mount('/content/drive')

# Verify the freshly-installed stack is loadable.
import vllm, lmcache, torch, transformers
print(f"vllm:         {vllm.__version__}")
print(f"lmcache:      {getattr(lmcache, '__version__', '(installed)')}")
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")

## 4. Clone project repo + install

If the repo already exists (re-running the notebook), this re-clones to get the latest commit.

In [ ]:
%cd /content
!rm -rf perceptual-mmcache
!git clone -b {BRANCH} {REPO_URL}
%cd /content/perceptual-mmcache
!pip install -q -e .

In [ ]:
# Initialize eval.paths from the workspace env var, then sanity-check
# unit tests pass before committing to long GPU runs.
from eval import paths
paths.set_workspace(WORKSPACE)
paths.ensure_dirs()
print(f"eval.paths.VIDEOS_DIR  = {paths.VIDEOS_DIR}")
print(f"eval.paths.QA_FILE     = {paths.QA_FILE}")
print(f"eval.paths.RESULTS_DIR = {paths.RESULTS_DIR}")

!pytest tests/ -q --tb=short

## 5. Prepare test dataset

Downloads a small Video-MME subset (short < 2 min clips) and writes `qa.jsonl` with MCQ options + letter answers. Skipped if videos already exist on Drive.

In [ ]:
# ====== Dataset prep config ======
N_VIDEOS = 10              # how many Video-MME clips to download
VIDEO_DURATION = "short"   # "short" (<2min) / "medium" / "long"
# =================================

import os
from eval.datasets.prepare_videomme import main as prepare_videomme

existing = [f for f in os.listdir(paths.VIDEOS_DIR)
            if f.endswith(('.mp4', '.mov', '.webm'))]

if len(existing) >= 5:
    print(f"Already have {len(existing)} videos in {paths.VIDEOS_DIR}, skipping prep.")
else:
    prepare_videomme(
        output_dir=paths.VIDEOS_DIR,
        n=N_VIDEOS,
        duration=VIDEO_DURATION,
        max_height=360,
    )

video_files = sorted(f for f in os.listdir(paths.VIDEOS_DIR)
                     if f.endswith(('.mp4', '.mov', '.webm')))
print(f"\nVideos ({len(video_files)}):")
for v in video_files:
    size_mb = os.path.getsize(f"{paths.VIDEOS_DIR}/{v}") / 1e6
    print(f"  - {v}  ({size_mb:.1f} MB)")

## 6. VLM smoke test

Loads Qwen3-VL-2B once and runs a single image inference to verify the stack is healthy before committing to long benchmark runs. First run downloads ~4 GB of weights.

In [ ]:
from eval.utils import smoke_test_vlm
smoke_test_vlm(model=MODEL, workspace=WORKSPACE)

## 7. Baseline benchmark

vLLM with bytewise `mm_hash` only — what every video-LLM deployment ships today. The runner now:
- builds MCQ prompts ("Answer with only the letter A/B/C/D") so accuracy is gradeable
- records `prompt_tokens`, `cached_tokens`, `kv_bytes_total/cached/recomputed` per call

JSONL goes to `paths.BASELINE_PATH` on Drive.

In [ ]:
from eval.run_baseline import run_baseline_benchmark

run_baseline_benchmark(
    videos_dir=paths.VIDEOS_DIR,
    qa_file=paths.QA_FILE,
    output_path=paths.BASELINE_PATH,
    model=MODEL,
    fps=1.0,
)

print(f"\n[ok] Baseline saved to: {paths.BASELINE_PATH}")

## 8. Perceptual benchmark

Same harness, with pmcache enabled — adjacent perceptually-similar frames get aliased to a single anchor `mm_hash` via EXIF tagging.

In [ ]:
from eval.run_perceptual import run_perceptual_benchmark

run_perceptual_benchmark(
    videos_dir=paths.VIDEOS_DIR,
    qa_file=paths.QA_FILE,
    output_path=paths.PERCEPTUAL_PATH,
    model=MODEL,
    tau=0.98,
    k=5,
    fps=1.0,
)

print(f"\n[ok] Perceptual saved to: {paths.PERCEPTUAL_PATH}")

## 9. Quick comparison

Side-by-side baseline vs perceptual: MCQ accuracy + KV cache footprint + TTFT + shim metrics. This is the headline result; the full report below adds plots.

In [ ]:
import json, statistics
from pathlib import Path

def _fmt_bytes(n):
    n = float(n)
    for u in ('B', 'KiB', 'MiB', 'GiB', 'TiB'):
        if n < 1024: return f'{n:.2f} {u}'
        n /= 1024
    return f'{n:.2f} PiB'

def _load(p):
    return [json.loads(l) for l in Path(p).read_text(encoding='utf-8').splitlines() if l.strip()]

b = _load(paths.BASELINE_PATH)
p = _load(paths.PERCEPTUAL_PATH)
print(f'rows: baseline={len(b)}  perceptual={len(p)}')

# Accuracy
b_correct = sum(1 for r in b if r.get('correct'))
p_correct = sum(1 for r in p if r.get('correct'))
print()
print(f'Accuracy: baseline={b_correct}/{len(b)}={b_correct/max(1,len(b))*100:.1f}%   '
      f'perceptual={p_correct}/{len(p)}={p_correct/max(1,len(p))*100:.1f}%')
agreed = sum(1 for rb, rp in zip(b,p) if rb.get('predicted_letter') == rp.get('predicted_letter'))
print(f'  same letter chosen on {agreed}/{min(len(b),len(p))} questions')

# KV cache
def s(rows, k): return sum(r.get(k, 0) for r in rows)
print()
print('KV cache footprint:')
for k in ('prompt_tokens', 'cached_tokens'):
    print(f'  {k:20s}: baseline={s(b,k):>10,}  perceptual={s(p,k):>10,}')
for k in ('kv_bytes_total', 'kv_bytes_cached', 'kv_bytes_recomputed'):
    bv, pv = s(b,k), s(p,k)
    delta = (pv - bv) / bv * 100 if bv else 0.0
    print(f'  {k:20s}: baseline={_fmt_bytes(bv):>12}  perceptual={_fmt_bytes(pv):>12}  ({delta:+.1f}%)')

saved = s(b,'kv_bytes_recomputed') - s(p,'kv_bytes_recomputed')
if saved > 0:
    print(f'  -> perceptual saved {_fmt_bytes(saved)} of KV recomputation')

# TTFT
if b and p:
    print()
    print(f'TTFT mean:  baseline={statistics.mean(r["ttft_s"] for r in b):.3f}s   '
          f'perceptual={statistics.mean(r["ttft_s"] for r in p):.3f}s')

# pmcache shim
if p:
    last = p[-1]
    print()
    print('pmcache shim (cumulative):')
    for k in ('tier1_hits', 'tier2_hits', 'tier2_rejects', 'misses', 'aliased_hashes', 'pmcache_hit_rate'):
        v = last.get(k)
        if isinstance(v, float):
            print(f'  {k:20s}: {v*100:.1f}%')
        else:
            print(f'  {k:20s}: {v}')

## 10. (Optional) Threshold sweep

Sweeps cosine threshold τ to find the operating point where accuracy is preserved and hit rate is maximized. Skip this cell if you just want a single-τ comparison.

In [ ]:
from eval.threshold_sweep import sweep_thresholds

sweep_thresholds(
    videos_dir=paths.VIDEOS_DIR,
    qa_file=paths.QA_FILE,
    taus=[0.95, 0.96, 0.97, 0.98, 0.99],
    output_dir=paths.SWEEP_DIR,
    model=MODEL,
    baseline_path=paths.BASELINE_PATH,
)

print(f"\n[ok] Sweep results saved to: {paths.SWEEP_DIR}")

## 11. Final report (plots + REPORT.md)

Aggregates baseline + perceptual + sweep into a markdown report with KV-bytes-saved, accuracy delta, and TTFT histograms.

In [ ]:
from eval.benchmark_videoqa import analyze_results

analyze_results(
    baseline_path=paths.BASELINE_PATH,
    perceptual_path=paths.PERCEPTUAL_PATH,
    sweep_dir=paths.SWEEP_DIR,
    output_dir=paths.REPORT_DIR,
)

print(f"\n[ok] Report saved to: {paths.REPORT_DIR}/REPORT.md")

In [ ]:
# Display report inline
from IPython.display import Markdown, display
with open(f'{paths.REPORT_DIR}/REPORT.md') as f:
    display(Markdown(f.read()))

In [ ]:
# Display the plot images
from IPython.display import Image
import os

for png in sorted(os.listdir(paths.REPORT_DIR)):
    if png.endswith('.png'):
        print(f"\n=== {png} ===")
        display(Image(filename=f'{paths.REPORT_DIR}/{png}'))

## 12. (Optional) Live demo with public link

Launches a Gradio app inside Colab and exposes a public share URL. Note: `demo/gradio_app.py` is currently stubbed (Step 13 of the build plan) — this cell will raise until that lands.

In [ ]:
from demo.gradio_app import launch_demo

launch_demo(
    model=MODEL,
    workspace=WORKSPACE,
    share=True,
)

---
## Troubleshooting

**`ImportError: cannot import name 'is_opaque_value' from 'torch._library.opaque_object'`** — vLLM and torch are out of sync because the kernel is still holding the preinstalled torch. Run the restart cell, then resume from the post-restart section.

**`AssertionError: Need >=20 GB VRAM`** — Runtime is on T4. Change to A100 or L4: `Runtime → Change runtime type → A100 GPU`.

**`ValueError: The decoder prompt (length N) is longer than the maximum model length`** — Should NOT happen on Qwen3-VL-2B (256K native context). If it does, pass `max_frames=24` to `run_baseline_benchmark` / `run_perceptual_benchmark` to cap frames per video.

**vLLM OOM during model load** — Lower `gpu_memory_utilization` by passing `vllm_kwargs={'gpu_memory_utilization': 0.75}` to the benchmark calls.

**`kv_bytes_per_token = 0` in output** — `eval.utils.kv_bytes_per_token` couldn't infer model dims. Open an issue with the model name; the helper walks `hf_config` + `text_config` + `llm_config` but new architectures may need another nested-config name.

**Perceptual run shows accuracy parity but ZERO KV bytes saved** — Expected without LMCache wired into vLLM's `kv_transfer_config`. vLLM's built-in prefix cache (`enable_prefix_caching=True`) only deduplicates contiguous prompt prefixes, so within-call mm_hash aliasing doesn't translate to cached_tokens. To convert aliasing into KV savings, the LLM needs to be constructed with an LMCache connector.

**Accuracy regression > 1pp in perceptual run** — τ is too loose. Re-run the threshold sweep, pick a τ where accuracy is on the plateau.

**Video-MME download fails / YouTube rate-limit** — Re-run the dataset cell; yt-dlp retries automatically. If it persists, lower `N_VIDEOS`. Videos that fail are silently skipped — check the printed file list.

**Session disconnected mid-benchmark** — Drive persisted everything up to the last completed JSONL write. Restart runtime, skip back to the post-restart resume section, re-run the cells you'd already completed (they're idempotent: Drive mount, eval.paths init, clone) then continue from where you left off.

**Gradio share link doesn't work** — Colab sometimes blocks tunnels. Use `share=False` and the local `gradio.live` link from the cell output.